In [ ]:
import pandas as pd
# Det her er en test
# Heeeej Viktor

## Load data
Energinet *ProductionConsumptionSettlement* (Energi Data Service), 2025, hourly, DK1 and DK2.  
The file uses `;` as separator and `,` as decimal mark.

In [2]:
df = pd.read_csv("ProductionConsumptionSettlement.csv", sep=";", decimal=",",
                 parse_dates=["HourUTC", "HourDK"])
df = df.sort_values(["HourUTC", "PriceArea"]).reset_index(drop=True)  # file is sorted newest first


df.drop(columns=["PriceArea","CentralPowerMWh","LocalPowerMWh","CommercialPowerMWh","LocalPowerSelfConMWh","HydroPowerMWh","ExchangeSE_MWh", "ExchangeGE_MWh"], inplace=True)  # drop columns that are not needed
df.head()

,HourUTC,HourDK,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,OnshoreWindGe50kW_MWh,SolarPowerLt10kW_MWh,SolarPowerGe10Lt40kW_MWh,SolarPowerGe40kW_MWh,SolarPowerSelfConMWh,UnknownProdMWh,ExchangeNO_MWh,ExchangeNL_MWh,ExchangeGB_MWh,ExchangeGreatBelt_MWh,GrossConsumptionMWh,GridLossTransmissionMWh,GridLossInterconnectorsMWh,GridLossDistributionMWh,PowerToHeatMWh
0,2024-12-31 23:00:00,2025-01-01 00:00:00,221.271215,1285.866796,14.123306,3269.386618,0.167449,0.055958,0.050220,0.0,0.145240,-1663.5741,-353.196,16.253469,-447.8,3068.153442,112.352214,62.3851,140.275622,824.219887
1,2024-12-31 23:00:00,2025-01-01 00:00:00,33.201800,897.953218,1.632528,601.122276,0.117938,0.022880,0.054689,0.0,2.745074,NaN,NaN,NaN,440.9,1684.124812,58.937666,1.9660,82.700911,139.791053
2,2025-01-01 00:00:00,2025-01-01 01:00:00,216.368038,1163.784669,13.996072,3235.545640,0.170201,0.054554,0.055574,0.0,0.307740,-1663.7999,-682.743,-250.274511,-457.8,3024.472823,119.134226,74.0599,138.148259,774.421902
3,2025-01-01 00:00:00,2025-01-01 01:00:00,26.030700,511.022869,1.680799,600.363627,0.123022,0.020321,0.011923,0.0,1.477600,NaN,NaN,NaN,450.7,1671.470494,44.998340,5.4900,82.648678,132.188810
4,2025-01-01 01:00:00,2025-01-01 02:00:00,73.050699,1025.820909,13.752589,3192.651473,0.151677,0.048444,0.051820,0.0,0.010471,-1641.8612,-111.974,-414.861463,16.8,3058.092913,98.579314,56.1772,137.748688,848.708904


## Simplified national time series (`df_simplyfi`)
Project guideline step 2: DK1 and DK2 contributions are added, so there is one row per hour for the whole of Denmark.  
We group on `HourUTC` because `HourDK` has a duplicated/missing hour at the daylight-saving shifts.

In [4]:
wind_cols = ["OffshoreWindLt100MW_MWh", "OffshoreWindGe100MW_MWh",
             "OnshoreWindLt50kW_MWh", "OnshoreWindGe50kW_MWh"]
solar_cols = ["SolarPowerLt10kW_MWh", "SolarPowerGe10Lt40kW_MWh",
              "SolarPowerGe40kW_MWh"]  # SolarPowerGe10kW_MWh is the sum of the two previous columns, but we include it for completeness

df["Wind_MWh"] = df[wind_cols].sum(axis=1)
df["Solar_MWh"] = df[solar_cols].sum(axis=1)

# Sum DK1 + DK2 for every hour
df_simplyfi = (df.groupby("HourUTC", as_index=False)
                 .agg(HourDK=("HourDK", "first"),
                      Wind_MWh=("Wind_MWh", "sum"),
                      Solar_MWh=("Solar_MWh", "sum"),
                      GrossConsumption_MWh=("GrossConsumptionMWh", "sum")))

df_simplyfi.drop(columns=["HourDK"], inplace=True)  # drop HourDK, as it is not unique for DK1+DK2
df_simplyfi.head()

,HourUTC,Wind_MWh,Solar_MWh,GrossConsumption_MWh
0,2024-12-31 23:00:00,6324.557757,0.469134,4752.278254
1,2025-01-01 00:00:00,5768.792414,0.435595,4695.943317
2,2025-01-01 01:00:00,5109.470665,0.397615,4635.561834
3,2025-01-01 02:00:00,5027.957554,0.404335,4605.831495
4,2025-01-01 03:00:00,5257.804306,0.646446,4446.709225


### Industry demand (constant)
Project guideline step 1: industry = 150 PJ + 30 PJ (data centres expected by 2030) = **180 PJ/year**, incl. agriculture.  
Assumption: industrial consumption is constant over the year, i.e. the same average value every hour.

$$\dot E_{\text{industry}} = \frac{180\ \text{PJ}}{3.6\cdot10^{-6}\ \text{PJ/MWh}\cdot 8760\ \text{h}} \approx 5708\ \text{MWh/h} \ (\approx 5.7\ \text{GW})$$

In [5]:
PJ_TO_MWH = 1e15 / 3.6e9          # 1 PJ = 277 777.8 MWh
industry_PJ_per_year = 150 + 30    # PJ/year (guideline step 1, incl. 30 PJ data centres)
hours_per_year = 8760              # 2025 is not a leap year

industry_MWh_per_hour = industry_PJ_per_year * PJ_TO_MWH / hours_per_year
df_simplyfi["Industry_MWh"] = industry_MWh_per_hour

print(f"Industry: {industry_PJ_per_year} PJ/yr = {industry_PJ_per_year*PJ_TO_MWH/1e6:.1f} TWh/yr "
      f"-> {industry_MWh_per_hour:.1f} MWh every hour ({industry_MWh_per_hour/1e3:.2f} GW)")
df_simplyfi.head()

Industry: 180 PJ/yr = 50.0 TWh/yr -> 5707.8 MWh every hour (5.71 GW)


,HourUTC,Wind_MWh,Solar_MWh,GrossConsumption_MWh,Industry_MWh
0,2024-12-31 23:00:00,6324.557757,0.469134,4752.278254,5707.762557
1,2025-01-01 00:00:00,5768.792414,0.435595,4695.943317,5707.762557
2,2025-01-01 01:00:00,5109.470665,0.397615,4635.561834,5707.762557
3,2025-01-01 02:00:00,5027.957554,0.404335,4605.831495,5707.762557
4,2025-01-01 03:00:00,5257.804306,0.646446,4446.709225,5707.762557


## Scaled 

In [32]:
Biomass = 200
Total_Future_PJ_per_year = 730   # PJ/year (guideline step 1, incl. 30 PJ data centres)
Future_elektrisety_PJ_per_year = Total_Future_PJ_per_year - Biomass 

Anual_Produktion_PJ = df['Wind_MWh'].sum() * 3.6e9 / 1e15 + df['Solar_MWh'].sum() * 3.6e9 / 1e15

scaling_factor = Future_elektrisety_PJ_per_year / Anual_Produktion_PJ 

scaling_factor

np.float64(6.3659832917925705)

In [34]:
df_simplyfi["ideal_Wind_MWh"] = scaling_factor*df_simplyfi["Wind_MWh"] 
df_simplyfi["ideal_Solar_MWh"] = scaling_factor*df_simplyfi["Solar_MWh"]
df_simplyfi["extra_available_electricity_MWh"] = (df_simplyfi["ideal_Wind_MWh"] + df_simplyfi["ideal_Solar_MWh"])- (df_simplyfi["GrossConsumption_MWh"])
df_simplyfi.head()


,HourUTC,Wind_MWh,Solar_MWh,GrossConsumption_MWh,Industry_MWh,ideal_Wind_MWh,ideal_Solar_MWh,extra_available_electricity_MWh
0,2024-12-31 23:00:00,6324.557757,0.469134,4752.278254,5707.762557,40262.029009,2.986499,35512.737254
1,2025-01-01 00:00:00,5768.792414,0.435595,4695.943317,5707.762557,36724.036121,2.772990,32030.865795
2,2025-01-01 01:00:00,5109.470665,0.397615,4635.561834,5707.762557,32526.804883,2.531210,27893.774260
3,2025-01-01 02:00:00,5027.957554,0.404335,4605.831495,5707.762557,32007.893781,2.573990,27404.636275
4,2025-01-01 03:00:00,5257.804306,0.646446,4446.709225,5707.762557,33471.094364,4.115264,29028.500403
